In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# 1. Install required packages
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    faiss-gpu-cu12 \
    pypdf \
    langchain \
    langchain-community \
    langchain-classic \
    langchain-huggingface \
    fastapi \
    uvicorn \
    python-multipart \
    pyngrok

In [ ]:
%%writefile server.py
import os
import re
import tempfile
import torch
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate

app = FastAPI(title="AI Document Intelligence Server", version="1.0")

# Enable CORS for local cross-origin client requests
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ----------------- Global State & Model Initialization -----------------
print("Loading embeddings and Mistral-7B LLM onto GPU...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# In-memory vector database reference
current_vectordb = None

# ----------------- Helper Functions -----------------
def generate_text(prompt: str, max_new_tokens: int = 600) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

def extract_clean_json(text: str) -> str:
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return f"```json\n{matches[-1]}\n```"
    return text

# ----------------- API Endpoints -----------------
@app.get("/")
def health_check():
    return {"status": "online", "gpu_available": torch.cuda.is_available()}

@app.post("/upload")
async def upload_pdf(file: UploadFile = File(...)):
    global current_vectordb
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        content = await file.read()
        tmp.write(content)
        tmp_path = tmp.name

    try:
        loader = PyPDFLoader(tmp_path)
        pages = loader.load()
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        chunks = splitter.split_documents(pages)
        current_vectordb = FAISS.from_documents(chunks, embeddings)
        return {"status": "success", "message": f"Successfully indexed {len(chunks)} chunks from {file.filename}"}
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

@app.get("/intelligence")
async def extract_intelligence():
    global current_vectordb
    if current_vectordb is None:
        return {"error": "No document has been uploaded or indexed yet."}

    schemas = [
        ResponseSchema(name="Document", description="Title or name of the document."),
        ResponseSchema(name="Summary", description="Executive summary of the document."),
        ResponseSchema(name="Key Points", description="List of primary points or critical findings."),
        ResponseSchema(name="Risks", description="Any stated risks, liabilities, constraints, or penalties."),
        ResponseSchema(name="References", description="Specific citations or section headings supporting findings.")
    ]
    parser = StructuredOutputParser.from_response_schemas(schemas)
    format_instructions = parser.get_format_instructions()

    retrieved = current_vectordb.similarity_search("overview academic structure admissions research student life", k=4)
    context = "\n\n".join([doc.page_content for doc in retrieved])

    prompt = f"""
You are an expert Document Intelligence System.
Analyze the provided context and extract the required structured sections based strictly on the text.

Context:
{context}

Respond ONLY in JSON format adhering strictly to this schema:
{format_instructions}
"""
    raw_response = generate_text(prompt, max_new_tokens=650)
    cleaned = extract_clean_json(raw_response)
    try:
        return parser.parse(cleaned)
    except Exception:
        return {"raw_response": raw_response}

@app.post("/ask")
async def answer_question(question: str = Form(...)):
    global current_vectordb
    if current_vectordb is None:
        return {"error": "No document has been uploaded or indexed yet."}

    qa_schemas = [
        ResponseSchema(name="answer", description="Clear and concise answer to the question."),
        ResponseSchema(name="sources", description="Direct citations, quotations, or section references.")
    ]
    qa_parser = StructuredOutputParser.from_response_schemas(qa_schemas)
    format_instructions = qa_parser.get_format_instructions()

    retrieved = current_vectordb.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in retrieved])

    prompt = f"""
You are an expert document assistant. Answer the user's question using ONLY the provided context.
Context:
{context}

Question:
{question}

Respond ONLY in JSON format following this exact schema:
{format_instructions}
"""
    raw_response = generate_text(prompt, max_new_tokens=400)
    cleaned = extract_clean_json(raw_response)
    try:
        return qa_parser.parse(cleaned)
    except Exception:
        return {"answer": raw_response, "sources": "Extracted from context chunks."}

In [ ]:
import subprocess
from pyngrok import ngrok

# 1. Kill any existing ngrok tunnels
ngrok.kill()

# 2. Set your ngrok token (Clean string without spaces or non-ASCII characters)
NGROK_TOKEN = "your-ngrok"
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Create public HTTPS tunnel to port 8000
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print("=" * 60)
print(f"👉 YOUR KAGGLE API URL: {public_url}")
print(f"👉 SWAGGER DOCS URL:   {public_url}/docs")
print("=" * 60)

# 4. Start the ASGI server
!uvicorn server:app --host 0.0.0.0 --port 8000